In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
#print(os.getenv("OPENAI_API_KEY"))

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=1.0)

In [22]:
from datasets import load_dataset
ds = load_dataset("cais/mmlu", "high_school_geography")

In [24]:
from langchain_community.agent_toolkits.load_tools import load_tools
from langgraph.prebuilt import create_react_agent


research_tools = load_tools(
  tool_names=["ddg-search", "arxiv", "wikipedia"],
  llm=llm
)

system_prompt = (
    "You're a hard-working, curious and creative student. "
    "You're working on exam quesion. Think step by step."
    "Always provide an argumentation for your answer. "
    "Do not assume anything, use available tools to search "
    "for evidence and supporting statements."
)


In [25]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langgraph.graph import MessagesState
from langgraph.prebuilt.chat_agent_executor import AgentState

raw_prompt_template = (
    "Answer the following multiple-choice question. "
    "\nQUESTION:\n{question}\n\nANSWER OPTIONS:\n{options}\n"
)
prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt),
     ("user", raw_prompt_template),
     ("placeholder", "{messages}")
     ]
)

class ResearchState(AgentState):
  question: str
  options: str

research_agent = create_react_agent(model=llm, tools=research_tools, state_schema=ResearchState, prompt=prompt)

/var/folders/74/56w7w3x17x79sh1vcbchn8600000gn/T/ipykernel_5389/1449410709.py:20: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(model=llm, tools=research_tools, state_schema=ResearchState, prompt=prompt)


In [26]:
i = 6
ds_dict = ds["test"].take(100).to_dict()
question = ds_dict["question"][i]
options = "\n".join([f"{i}. {a}" for i, a in enumerate(ds_dict["choices"][i])])

In [27]:
async for _, event in research_agent.astream({"question": question, "options": options}, stream_mode=["values"]):
  print(len(event["messages"]))

0
1
2
3
4
5
6
7
8
9


In [7]:
async for _, event in research_agent.astream({"question": question, "options": options}, stream_mode=["updates"]):
  node = list(event.keys())[0]
  print(node, len(event[node].get("messages", [])))

agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1
tools 1
agent 1


In [28]:
seen_events = set([])
async for event in research_agent.astream_events({"question": question, "options": options}, version="v1"):
  if event["event"] not in seen_events:
    seen_events.add(event["event"])

print(seen_events)

{'on_prompt_end', 'on_tool_start', 'on_chat_model_end', 'on_tool_end', 'on_chat_model_stream', 'on_chain_start', 'on_chain_end', 'on_chat_model_start', 'on_prompt_start', 'on_chain_stream'}


In [29]:
# ---------------------------------------------------------------------------
# 2. Pull a question from MMLU to feed the agent
# ---------------------------------------------------------------------------
 
ds = load_dataset("cais/mmlu", "high_school_geography")
ds_dict = ds["test"].take(100).to_dict()
 
i = 6
question = ds_dict["question"][i]
options = "\n".join([f"{i}. {a}" for i, a in enumerate(ds_dict["choices"][i])])
 
print(f"QUESTION: {question}")
print(f"OPTIONS:\n{options}\n")
print("=" * 70)

QUESTION: Which of the following countries does NOT have a well-known example of a relict boundary?
OPTIONS:
0. Vietnam
1. United Kingdom
2. Germany
3. Bolivia



In [30]:
# ---------------------------------------------------------------------------
# 3. Streaming Mode 1: "values" - full state after each step
# ---------------------------------------------------------------------------
# Use this when you want to re-render the entire conversation each turn
# (e.g. a chat UI that redraws on every update).
 
async def demo_values_mode():
    print("\n[ values mode ] -- full state after each node\n")
    async for _, event in research_agent.astream(
        {"question": question, "options": options},
        stream_mode=["values"],
    ):
        # event["messages"] is the FULL conversation history at this point
        print(f"  total messages in state: {len(event['messages'])}")
    # Expected progression: 0 -> 1 -> 5 -> 6
    #   0: initial state
    #   1: agent emitted AIMessage with 4 parallel tool calls
    #   5: tools node added 4 ToolMessages (1 + 4 = 5)
    #   6: agent emitted final AIMessage synthesizing results (5 + 1 = 6)
 
 

In [31]:
# ---------------------------------------------------------------------------
# 4. Streaming Mode 2: "updates" - per-node diffs
# ---------------------------------------------------------------------------
# Use this for progress indicators / per-node logging.
# Event shape: {node_name: {field: new_value}}
 
async def demo_updates_mode_compact():
    print("\n[ updates mode (compact) ] -- which node ran, how many msgs added\n")
    async for _, event in research_agent.astream(
        {"question": question, "options": options},
        stream_mode=["updates"],
    ):
        node = list(event.keys())[0]
        msg_count = len(event[node].get("messages", []))
        print(f"  node={node!r}  messages_added={msg_count}")
    # Expected: agent 1 -> tools 1 -> agent 1
    # NOTE: the "tools 1" event has a single update payload that contains
    # 4 ToolMessages inside (parallel tool calls collapse into one update).
 
 
async def demo_updates_mode_full():
    print("\n[ updates mode (full payload) ] -- complete dict each step\n")
    async for _, event in research_agent.astream(
        {"question": question, "options": options},
        stream_mode=["updates"],
    ):
        print(event)
        print("-" * 70)

In [34]:
# ---------------------------------------------------------------------------
# 5. Streaming Mode 3: astream_events - fine-grained events
# ---------------------------------------------------------------------------
# Use this for token-by-token streaming, tracing (Langfuse/LangSmith),
# or any custom hook below the node level.
 
async def demo_astream_events():
    print("\n[ astream_events v1 ] -- what event types fire?\n")
    seen_events = set()
    async for event in research_agent.astream_events(
        {"question": question, "options": options},
        version="v1",
    ):
        seen_events.add(event["event"])
    print(f"  unique event types: {seen_events}")
    # Expected events include:
    #   on_prompt_start / on_prompt_end           -- prompt template
    #   on_chat_model_start / on_chat_model_stream / on_chat_model_end
    #                                             -- LLM calls (stream = per-token chunks)
    #   on_tool_start / on_tool_end               -- each tool invocation
    #   on_chain_start / on_chain_stream / on_chain_end
    #                                             -- the wrapping chains
 
 
async def demo_astream_events_token_streaming():
    """
    Practical example: stream LLM tokens as they're generated.
    This is what powers a "typing" effect in chat UIs.
    """
    print("\n[ astream_events -- token streaming ]\n")
    async for event in research_agent.astream_events(
        {"question": question, "options": options},
        version="v1",
    ):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"]
            # chunk is an AIMessageChunk; .content is the text piece
            if chunk.content:
                print(chunk.content, end="", flush=True)
        elif event["event"] == "on_tool_start":
            print(f"\n\n[calling tool: {event['name']}]\n")
        elif event["event"] == "on_tool_end":
            print(f"\n[tool finished: {event['name']}]\n")
    print()
 
 
# ---------------------------------------------------------------------------
# 6. Run all demos
# ---------------------------------------------------------------------------
 
async def main():
    await demo_values_mode()
    await demo_updates_mode_compact()
    await demo_updates_mode_full()
    await demo_astream_events()
    await demo_astream_events_token_streaming()
 
import asyncio
 


In [36]:
await demo_values_mode()


[ values mode ] -- full state after each node

  total messages in state: 0
  total messages in state: 1
  total messages in state: 2
  total messages in state: 3
  total messages in state: 4
  total messages in state: 5
  total messages in state: 6
  total messages in state: 7
  total messages in state: 8
  total messages in state: 9
